In [0]:
CREATE OR REPLACE PROCEDURE dbw_healthcare_de_dev.etl.usp_retry_file
(
    IN p_file_id STRING,
    IN p_new_adf_run_id STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

    UPDATE dbw_healthcare_de_dev.etl.file_control
    SET
        pipeline_status = 'IN_PROGRESS',
        adf_run_id = p_new_adf_run_id,
        processing_start_ts = CURRENT_TIMESTAMP(),
        processing_end_ts = NULL,
        error_message = NULL,
        last_updated_ts = CURRENT_TIMESTAMP()

    WHERE file_id = p_file_id
      AND pipeline_status = 'FAILED'
      AND processing_stage IN ('BRONZE', 'SILVER', 'GOLD')
      AND is_enabled = TRUE
      AND retry_count < 3
      AND p_new_adf_run_id IS NOT NULL
      AND p_new_adf_run_id <> adf_run_id;

END;